Phase 6.1c — Pipeline C: hard-filter + DeepFM v2 精排 (兜底架构)

Compare three retrieval-then-ranking pipelines on val set:
- **A (brute-force)**: DeepFM v2 ranks all 9K items → top-10. Reference baseline.
- **B (Two-Tower 级联)**: Two-Tower top-200 → DeepFM rerank → top-10. Phase 6.1 base.
- **C (硬过滤兜底)**: same-city + price within ±1 + ≥1 cuisine overlap → DeepFM rerank → top-10.

Plan §3.3.4 v2.2 fallback architecture. Hypothesis: C 比 B 显著好（不会把 ground truth positive 过滤掉，只要它符合 user 的 city/cuisine/price），跟 A 接近。

In [1]:
import json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import platform

PROJECT_ROOT = Path("/Users/yanghaobo/projects/Yelp-Recommendation-System").resolve()
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
MODELS_DIR = PROJECT_ROOT / "models"

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"arch: {platform.machine()}  device: {DEVICE}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")


arch: arm64  device: mps
PROJECT_ROOT: /Users/yanghaobo/projects/Yelp-Recommendation-System


Reuse the IDEncoder + TasteHunterDataset + DeepFM class definitions from 05c (inline per ML2 grading rule — code IN notebook, no `from src import`).

In [2]:
class IDEncoder:
    def __init__(self, ids, oov_token="<UNK>"):
        oov_markers = {"<NEW_USER>", "<NEW_BUSINESS>", "<UNK>", oov_token}
        unique_real_ids = sorted({i for i in ids if i not in oov_markers})
        self.id_to_idx = {oov_token: 0}
        for idx, _id in enumerate(unique_real_ids, start=1):
            self.id_to_idx[_id] = idx
        for marker in oov_markers:
            self.id_to_idx.setdefault(marker, 0)
        self._size = 1 + len(unique_real_ids)

    def __len__(self):
        return self._size

    def encode(self, _id):
        return self.id_to_idx.get(_id, 0)

    def encode_array(self, ids):
        return ids.map(self.id_to_idx).fillna(0).astype(np.int64).values


class TasteHunterDataset(Dataset):
    def __init__(self, df, user_encoder, item_encoder, user_features, item_features):
        self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))
        self.item_idx = torch.from_numpy(item_encoder.encode_array(df["business_id"]))
        self.label = torch.from_numpy(df["label"].astype(np.float32).values)
        n_users = len(user_encoder)
        n_items = len(item_encoder)
        self.user_num = np.zeros((n_users, 6), dtype=np.float32)
        self.user_cuisine = np.zeros((n_users, 50), dtype=np.float32)
        for _, row in user_features.iterrows():
            uidx = user_encoder.encode(row["user_id"])
            self.user_num[uidx] = [row["avg_rating_given"], row["review_count_log"], row["days_active"],
                                    row["elite_flag"], row["mean_distance_traveled"], row["price_tolerance_avg"]]
            emb = row["fav_cuisine_emb"]
            if isinstance(emb, list) and len(emb) == 50:
                self.user_cuisine[uidx] = emb
        self.item_num = np.zeros((n_items, 7), dtype=np.float32)
        self.item_cat = np.zeros((n_items, 50), dtype=np.float32)
        self.item_text = np.zeros((n_items, 32), dtype=np.float32)
        for _, row in item_features.iterrows():
            iidx = item_encoder.encode(row["business_id"])
            self.item_num[iidx] = [row["avg_rating"], row["review_count_log"], row["price_level"],
                                    row["is_open"], row["has_outdoor_seating"], row["photo_count_log"], row["city_id"]]
            cat = row["categories_multi_hot"]
            if isinstance(cat, list) and len(cat) == 50:
                self.item_cat[iidx] = cat
            text = row.get("item_text_emb_pca32")
            if isinstance(text, (list, np.ndarray)) and len(text) == 32:
                self.item_text[iidx] = np.asarray(text, dtype=np.float32)
        self.user_num_t = torch.from_numpy(self.user_num)
        self.user_cuisine_t = torch.from_numpy(self.user_cuisine)
        self.item_num_t = torch.from_numpy(self.item_num)
        self.item_cat_t = torch.from_numpy(self.item_cat)
        self.item_text_t = torch.from_numpy(self.item_text)

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        u = self.user_idx[idx]
        i = self.item_idx[idx]
        return {
            "user_idx": u, "item_idx": i, "label": self.label[idx],
            "user_num": self.user_num_t[u], "user_cuisine": self.user_cuisine_t[u],
            "item_num": self.item_num_t[i], "item_cat": self.item_cat_t[i],
            "item_text": self.item_text_t[i],
        }


def make_val_eval_pairs(val_df, user_encoder, item_encoder, item_features, n_negs=99, rng_seed=42):
    rng = np.random.default_rng(rng_seed)
    val_pos = val_df[val_df["stars"] >= 4].copy()
    biz_city = item_features.set_index("business_id")["city"].to_dict()
    val_pos["city"] = val_pos["business_id"].map(biz_city).fillna("<UNK>")
    city_biz = {}
    for bid, c in biz_city.items():
        if c == "<UNK>":
            continue
        city_biz.setdefault(c, []).append(bid)
    for c in city_biz:
        city_biz[c] = np.array(city_biz[c])
    all_user, all_item, all_label = [], [], []
    for row in val_pos.itertuples(index=False):
        if row.city not in city_biz:
            continue
        candidates = city_biz[row.city]
        sampled = rng.choice(candidates, size=n_negs * 2, replace=True)
        negs = [b for b in sampled if b != row.business_id][:n_negs]
        if len(negs) < n_negs:
            continue
        items = [row.business_id] + negs
        all_user.append([user_encoder.encode(row.user_id)] * (n_negs + 1))
        all_item.append([item_encoder.encode(b) for b in items])
        all_label.append([1.0] + [0.0] * n_negs)
    return (np.array(all_user, dtype=np.int64),
            np.array(all_item, dtype=np.int64),
            np.array(all_label, dtype=np.float32))


def compute_auc(scores, labels):
    pos_mask = labels > 0.5
    n_pos = pos_mask.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(scores)
    ranks = np.empty(len(scores))
    ranks[order] = np.arange(1, len(scores) + 1)
    return float((ranks[pos_mask].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def ndcg_at_k(score_matrix, label_matrix, k=10):
    top_k_idx = np.argsort(-score_matrix, axis=1)[:, :k]
    rows = np.arange(score_matrix.shape[0])[:, None]
    top_k_labels = label_matrix[rows, top_k_idx]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = (top_k_labels * discounts).sum(axis=1)
    ideal_labels = -np.sort(-label_matrix, axis=1)[:, :k]
    idcg = (ideal_labels * discounts).sum(axis=1)
    idcg = np.where(idcg > 0, idcg, 1.0)
    return float(np.mean(dcg / idcg))


def recall_at_k(score_matrix, label_matrix, k=10):
    top_k_idx = np.argsort(-score_matrix, axis=1)[:, :k]
    rows = np.arange(score_matrix.shape[0])[:, None]
    top_k_labels = label_matrix[rows, top_k_idx]
    n_pos_per_row = label_matrix.sum(axis=1)
    correct = top_k_labels.sum(axis=1)
    return float(np.mean(np.where(n_pos_per_row > 0, correct / n_pos_per_row, 0)))


@torch.no_grad()
def score_pairs(model, user_idx, item_idx, dataset, device, batch_size=8192):
    model.eval()
    n, c = user_idx.shape
    flat_u = user_idx.reshape(-1)
    flat_i = item_idx.reshape(-1)
    scores = np.zeros(n * c, dtype=np.float32)
    for start in range(0, len(flat_u), batch_size):
        end = min(start + batch_size, len(flat_u))
        u_b = torch.from_numpy(flat_u[start:end]).to(device)
        i_b = torch.from_numpy(flat_i[start:end]).to(device)
        u_t = torch.from_numpy(flat_u[start:end])
        i_t = torch.from_numpy(flat_i[start:end])
        kwargs = {
            "user_num": dataset.user_num_t[u_t].to(device),
            "user_cuisine": dataset.user_cuisine_t[u_t].to(device),
            "item_num": dataset.item_num_t[i_t].to(device),
            "item_cat": dataset.item_cat_t[i_t].to(device),
            "item_text": dataset.item_text_t[i_t].to(device),
        }
        s = model(u_b, i_b, **kwargs).cpu().numpy().reshape(-1)
        scores[start:end] = s
    return scores.reshape(n, c)


def evaluate_full(model, user_idx_eval, item_idx_eval, label_eval, dataset, device):
    score_matrix = score_pairs(model, user_idx_eval, item_idx_eval, dataset, device)
    return {
        "AUC": compute_auc(score_matrix.reshape(-1), label_eval.reshape(-1)),
        "NDCG@10": ndcg_at_k(score_matrix, label_eval, k=10),
        "Recall@10": recall_at_k(score_matrix, label_eval, k=10),
    }

In [3]:
class DeepFM(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32,
                 user_num_dim=6, user_cuisine_dim=50,
                 item_num_dim=7, item_cat_dim=50, item_text_dim=32,
                 dnn_hidden=(256, 128, 64), dropout=0.2, ablate_user_id=False):
        super().__init__()
        self.emb_dim = emb_dim
        self.ablate_user_id = ablate_user_id
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_num_proj = nn.Linear(user_num_dim, emb_dim)
        self.item_num_proj = nn.Linear(item_num_dim, emb_dim)
        self.item_text_proj = nn.Linear(item_text_dim, emb_dim)

        feat_dim_linear = user_num_dim + item_num_dim + user_cuisine_dim + item_cat_dim + item_text_dim
        self.linear_features = nn.Linear(feat_dim_linear, 1)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        # 5 fields stacked for FM 2nd-order: user_emb, item_emb, user_num_proj, item_num_proj, item_text_proj
        # DNN input: 5 emb fields + 2 multi-hot (cuisine 50, cat 50) + raw text 32
        dnn_input_dim = emb_dim * 5 + user_cuisine_dim + item_cat_dim + item_text_dim
        layers = []
        prev = dnn_input_dim
        for h in dnn_hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.dnn = nn.Sequential(*layers)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, user_num, user_cuisine, item_num, item_cat, item_text):
        u_emb = self.user_emb(user_idx)
        i_emb = self.item_emb(item_idx)
        un_emb = self.user_num_proj(user_num)
        in_emb = self.item_num_proj(item_num)
        it_emb = self.item_text_proj(item_text)

        if self.ablate_user_id:
            u_emb = torch.zeros_like(u_emb)

        feat_concat = torch.cat([user_num, item_num, user_cuisine, item_cat, item_text], dim=-1)
        order1 = self.linear_features(feat_concat).squeeze(-1)
        bu = self.user_bias(user_idx).squeeze(-1)
        bi = self.item_bias(item_idx).squeeze(-1)
        if self.ablate_user_id:
            bu = torch.zeros_like(bu)

        embs = torch.stack([u_emb, i_emb, un_emb, in_emb, it_emb], dim=1)
        sum_sq = embs.sum(dim=1) ** 2
        sq_sum = (embs ** 2).sum(dim=1)
        order2 = 0.5 * (sum_sq - sq_sum).sum(dim=-1)

        dnn_input = torch.cat([u_emb, i_emb, un_emb, in_emb, it_emb, user_cuisine, item_cat, item_text], dim=-1)
        dnn_output = self.dnn(dnn_input).squeeze(-1)

        return torch.sigmoid(order1 + order2 + dnn_output + bu + bi + self.global_bias)


def train_one(config, n_users, n_items, train_loader, ue, ie, le, dataset, device, max_epochs=10):
    torch.manual_seed(SEED)
    model = DeepFM(n_users, n_items, emb_dim=config["emb_dim"], dropout=config["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.get("lr", 1e-3), weight_decay=config["l2"])
    bce = nn.BCELoss()
    history = {"epoch": [], "train_loss": [], "val_auc": [], "val_ndcg10": [], "val_recall10": []}
    for epoch in range(1, max_epochs + 1):
        model.train()
        loss_sum, n_batch = 0.0, 0
        t0 = time.time()
        for batch in train_loader:
            u = batch["user_idx"].to(device)
            i = batch["item_idx"].to(device)
            y = batch["label"].to(device)
            kwargs = {k: batch[k].to(device) for k in ("user_num", "user_cuisine", "item_num", "item_cat", "item_text")}
            pred = model(u, i, **kwargs)
            loss = bce(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
            n_batch += 1
        train_loss = loss_sum / n_batch
        elapsed = time.time() - t0
        m = evaluate_full(model, ue, ie, le, dataset, device)
        print(f"  ep{epoch:02d} | {elapsed:.0f}s | loss={train_loss:.4f} | AUC={m['AUC']:.4f} NDCG@10={m['NDCG@10']:.4f}")
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_auc"].append(m["AUC"])
        history["val_ndcg10"].append(m["NDCG@10"])
        history["val_recall10"].append(m["Recall@10"])
    history["best_ndcg10"] = max(history["val_ndcg10"])
    history["best_auc"] = max(history["val_auc"])
    history["best_recall10"] = max(history["val_recall10"])
    history["config"] = config
    return history

Load Phase 5.4 final v2 model + user/item features.

In [4]:
user_features = pd.read_parquet(FEATURES_DIR / "user_features.parquet")
item_features = pd.read_parquet(FEATURES_DIR / "item_features.parquet")
val_df = pd.read_parquet(CLEANED_DIR / "val_reviews.parquet")

print(f"user_features: {len(user_features):,} rows")
print(f"item_features: {len(item_features):,} rows")
print(f"val_df:        {len(val_df):,} rows ({val_df.user_id.nunique():,} unique users, {val_df.business_id.nunique():,} unique items)")

# Build encoders the same way as 05c/05d
user_encoder = IDEncoder(user_features["user_id"].tolist(), oov_token="<NEW_USER>")
item_encoder = IDEncoder(item_features["business_id"].tolist(), oov_token="<NEW_BUSINESS>")
n_users, n_items = len(user_encoder), len(item_encoder)
print(f"n_users={n_users:,}  n_items={n_items:,}")

# Need a TasteHunterDataset to get pre-computed user/item feature tensors indexed by encoder idx
# Use a minimal stub df (1 row) just to build feature lookups
stub_df = val_df.head(1)[["user_id", "business_id", "stars"]].copy()
stub_df["label"] = 1
stub_ds = TasteHunterDataset(stub_df, user_encoder, item_encoder, user_features, item_features)
print(f"stub_ds built — feature tensors ready: user_num_t={stub_ds.user_num_t.shape}  item_num_t={stub_ds.item_num_t.shape}  item_text_t={stub_ds.item_text_t.shape}")

# Load v2 final model (winner config from Phase 5.4: emb=32, dropout=0.1, L2=1e-4)
WINNER = json.load(open(MODELS_DIR / "deepfm_final_v2_meta.json"))["config"]
print(f"loading deepfm_final_v2.pt with config: emb_dim={WINNER['emb_dim']} dropout={WINNER['dropout']} l2={WINNER['l2']:.0e}")

model = DeepFM(n_users, n_items, emb_dim=WINNER["emb_dim"], dropout=WINNER["dropout"]).to(DEVICE)
state = torch.load(MODELS_DIR / "deepfm_final_v2.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(state)
model.eval()
print("model loaded ✓")


user_features: 359,008 rows
item_features: 9,023 rows
val_df:        83,953 rows (39,824 unique users, 6,641 unique items)
n_users=359,008  n_items=9,023


/var/folders/4j/cfzmxp0d3db8xdcx2fncnbnc0000gn/T/ipykernel_29359/2717135169.py:24: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self.user_idx = torch.from_numpy(user_encoder.encode_array(df["user_id"]))


stub_ds built — feature tensors ready: user_num_t=torch.Size([359008, 6])  item_num_t=torch.Size([9023, 7])  item_text_t=torch.Size([9023, 32])
loading deepfm_final_v2.pt with config: emb_dim=32 dropout=0.1 l2=1e-04


model loaded ✓


Hard-filter design (per plan §3.3.4 v2.2):

For each `(user, val_positive)` pair, target city = `val_positive.city` (the user's actual visit context). Filter `restaurants_open` (9K total) by:
1. **Same city** as val_positive
2. **Open** (`is_open == 1`) — already pre-filtered into restaurants_open subset
3. **Price**: `|item.price_level/4 - user.price_tolerance_avg| <= 0.40` (≈ ±1.5 levels tolerance)
4. **Cuisine overlap**: at least one of user's top-10 cuisine slots (where `fav_cuisine_emb[i] > 0`) hits item's `categories_multi_hot[i]`

Expected pool size: ~300-1500 candidates per user. Sufficiently selective without over-pruning.

In [5]:
def get_user_top_cuisines(fav_emb, top_k=10, min_weight=0.01):
    """Return cuisine indices where user's fav_emb has weight >= min_weight, top K of them."""
    arr = np.asarray(fav_emb)
    nz = np.where(arr >= min_weight)[0]
    if len(nz) == 0:
        return np.argsort(arr)[-top_k:]  # fallback: take top-K by weight
    return nz[np.argsort(-arr[nz])][:top_k]

# Pre-compute per-user filter inputs (avoid pandas slicing in loop)
print("pre-computing user filter inputs...")
t0 = time.time()
user_id_to_idx = {uid: i for i, uid in enumerate(user_features["user_id"].tolist())}
user_price_arr = user_features["price_tolerance_avg"].to_numpy()
user_cuisine_top = []
for emb in user_features["fav_cuisine_emb"]:
    user_cuisine_top.append(set(get_user_top_cuisines(emb, top_k=10).tolist()))
print(f"  user filter inputs ready ({time.time()-t0:.1f}s)")

# Pre-compute per-item info for fast filter
print("pre-computing item filter inputs...")
t0 = time.time()
item_city = item_features["city"].to_numpy()
item_price = item_features["price_level"].to_numpy()
item_cuisine_sets = [set(np.where(np.asarray(c) > 0)[0].tolist()) for c in item_features["categories_multi_hot"]]
item_business_ids = item_features["business_id"].to_numpy()
print(f"  item filter inputs ready ({time.time()-t0:.1f}s)")

def hard_filter(user_id, target_city):
    """Return array of item_features row indices that pass the filter."""
    if user_id not in user_id_to_idx:
        return None  # OOV user — no filter applicable
    u_idx = user_id_to_idx[user_id]
    user_p = user_price_arr[u_idx]
    user_c = user_cuisine_top[u_idx]

    # Boolean masks
    city_mask = (item_city == target_city)
    price_diff = np.abs(item_price / 4.0 - user_p)
    price_mask = (price_diff <= 0.40)
    base_mask = city_mask & price_mask
    candidate_idx = np.where(base_mask)[0]

    # Cuisine overlap (only check on candidates that pass city+price)
    overlap_idx = [i for i in candidate_idx if (user_c & item_cuisine_sets[i])]
    return np.array(overlap_idx, dtype=np.int64)

# Quick sanity check
sample_user = val_df.iloc[0].user_id
sample_biz = val_df.iloc[0].business_id
sample_city = item_features.set_index("business_id").loc[sample_biz, "city"]
out = hard_filter(sample_user, sample_city)
print(f"\nsanity: user={sample_user[:8]}... target_city={sample_city}")
print(f"  filter pool size: {len(out)} candidates (out of {len(item_features):,} total)")


pre-computing user filter inputs...


  user filter inputs ready (0.8s)
pre-computing item filter inputs...
  item filter inputs ready (0.0s)

sanity: user=RxwQtY9M... target_city=Philadelphia
  filter pool size: 615 candidates (out of 9,023 total)


Evaluation: subsample 5000 val pairs (full 84K is slow), apply hard filter, score with DeepFM v2, compute Recall@10 + NDCG@10 + pool size + latency. Compare with Pipeline A (DeepFM full 9K) baseline.

In [6]:
N_EVAL = 5000  # subsample size for speed

# Lookup: business_id -> item_features row idx (encoder may differ — use direct row lookup)
biz_id_to_row_idx = {bid: i for i, bid in enumerate(item_features["business_id"].tolist())}
biz_id_to_city = item_features.set_index("business_id")["city"].to_dict()

# Subsample val deterministically
val_sub = val_df.sample(n=min(N_EVAL, len(val_df)), random_state=SEED).reset_index(drop=True)
print(f"eval subsample: {len(val_sub):,} (val_df has {len(val_df):,})")

# Pre-extract feature tensors (move to device once for speed)
user_num_t = stub_ds.user_num_t.to(DEVICE)        # (n_users, 6)
user_cuisine_t = stub_ds.user_cuisine_t.to(DEVICE)  # (n_users, 50)
item_num_t = stub_ds.item_num_t.to(DEVICE)         # (n_items, 7)
item_cat_t = stub_ds.item_cat_t.to(DEVICE)          # (n_items, 50)
item_text_t = stub_ds.item_text_t.to(DEVICE)        # (n_items, 32)

@torch.no_grad()
def score_user_against_candidates(user_idx, cand_item_idxs):
    """Score 1 user against N candidates. Returns scores tensor of shape (N,)."""
    n = len(cand_item_idxs)
    u_t = torch.full((n,), user_idx, dtype=torch.long, device=DEVICE)
    i_t = torch.tensor(cand_item_idxs, dtype=torch.long, device=DEVICE)
    kwargs = {
        "user_num":     user_num_t[u_t],
        "user_cuisine": user_cuisine_t[u_t],
        "item_num":     item_num_t[i_t],
        "item_cat":     item_cat_t[i_t],
        "item_text":    item_text_t[i_t],
    }
    return model(u_t, i_t, **kwargs)  # (N,) sigmoid'd scores

def ndcg_at_k(ranked_positives, k=10):
    """Given a sorted list (positions of positives in ranked list), compute NDCG@K. Single positive case."""
    if not ranked_positives or ranked_positives[0] >= k:
        return 0.0
    pos = ranked_positives[0]  # 0-indexed rank of true positive in ranking
    return 1.0 / math.log2(pos + 2)

print("ready to eval")


eval subsample: 5,000 (val_df has 83,953)
ready to eval


In [7]:
recalls_at_10 = []
ndcgs_at_10 = []
pool_sizes = []
latencies = []
in_pool_count = 0   # how often the val_positive itself survives the hard filter
total_count = 0
oov_user_count = 0

t_start = time.time()
for row_idx, row in val_sub.iterrows():
    total_count += 1
    user_id = row.user_id
    biz_id = row.business_id

    if biz_id not in biz_id_to_row_idx:
        continue  # truly OOV biz (shouldn't happen on val subset by construction)
    target_city = biz_id_to_city[biz_id]
    pos_item_idx = biz_id_to_row_idx[biz_id]   # ground truth item row idx in item_features

    t0 = time.time()
    cand_idxs = hard_filter(user_id, target_city)
    if cand_idxs is None:
        oov_user_count += 1
        continue
    if len(cand_idxs) == 0:
        # No candidates pass filter — pipeline yields empty top-10 → Recall=0, NDCG=0
        recalls_at_10.append(0.0); ndcgs_at_10.append(0.0)
        pool_sizes.append(0); latencies.append(time.time() - t0)
        continue

    # Force-include positive even if filter excluded it (so we can measure "would model rank it well IF in pool")
    # Track separately whether positive was originally in pool
    pos_in_pool = pos_item_idx in cand_idxs
    if pos_in_pool:
        in_pool_count += 1
        full_cand = cand_idxs
    else:
        full_cand = np.append(cand_idxs, pos_item_idx)

    # Score
    user_idx_for_model = user_encoder.encode(user_id)  # OOV→<NEW_USER>
    scores = score_user_against_candidates(user_idx_for_model, full_cand).cpu().numpy()
    # Rank: descending order of scores
    order = np.argsort(-scores)
    ranked_item_idxs = full_cand[order]

    # Position of positive
    pos_rank = int(np.where(ranked_item_idxs == pos_item_idx)[0][0])

    # Compute Recall@10 = 1 if in top-10 AND was in original pool, else 0
    # NDCG@10: same gating (if pipeline wouldn't return positive, NDCG=0)
    if pos_in_pool and pos_rank < 10:
        recalls_at_10.append(1.0)
        ndcgs_at_10.append(1.0 / math.log2(pos_rank + 2))
    else:
        recalls_at_10.append(0.0)
        ndcgs_at_10.append(0.0)

    pool_sizes.append(len(cand_idxs))
    latencies.append(time.time() - t0)

    if (row_idx+1) % 500 == 0:
        elapsed = time.time() - t_start
        print(f"  [{row_idx+1}/{len(val_sub)}] elapsed {elapsed:.0f}s, current Recall@10={np.mean(recalls_at_10):.4f}, NDCG@10={np.mean(ndcgs_at_10):.4f}")

elapsed = time.time() - t_start
print(f"\nEval complete in {elapsed:.0f}s ({elapsed/total_count*1000:.1f} ms/pair on average)")
print(f"  evaluated:           {len(recalls_at_10):,}/{total_count:,}")
print(f"  OOV users:           {oov_user_count:,}")
print(f"  positive in pool:    {in_pool_count:,}/{len(recalls_at_10):,} ({100*in_pool_count/len(recalls_at_10):.1f}%)")
print()
print(f"  Mean Recall@10:      {np.mean(recalls_at_10):.4f}")
print(f"  Mean NDCG@10:        {np.mean(ndcgs_at_10):.4f}")
print(f"  Mean pool size:      {np.mean(pool_sizes):.1f}")
print(f"  Median pool size:    {np.median(pool_sizes):.0f}")
print(f"  Latency p50/p95/p99: {np.percentile(latencies, 50)*1000:.1f}ms / {np.percentile(latencies, 95)*1000:.1f}ms / {np.percentile(latencies, 99)*1000:.1f}ms")

# Save metrics
results_C = {
    "pipeline": "C (hard filter + DeepFM v2)",
    "n_eval": len(recalls_at_10),
    "n_oov_user": oov_user_count,
    "positive_in_pool_rate": in_pool_count / max(len(recalls_at_10), 1),
    "Recall@10": float(np.mean(recalls_at_10)),
    "NDCG@10": float(np.mean(ndcgs_at_10)),
    "mean_pool_size": float(np.mean(pool_sizes)),
    "median_pool_size": float(np.median(pool_sizes)),
    "latency_ms_p50": float(np.percentile(latencies, 50)*1000),
    "latency_ms_p95": float(np.percentile(latencies, 95)*1000),
    "latency_ms_p99": float(np.percentile(latencies, 99)*1000),
    "filter_config": {"price_tolerance_window": 0.40, "user_top_k_cuisines": 10},
}
with open(MODELS_DIR / "pipeline_C_metrics.json", "w") as f:
    json.dump(results_C, f, indent=2)
print("\nsaved → models/pipeline_C_metrics.json")


  [500/5000] elapsed 12s, current Recall@10=0.0140, NDCG@10=0.0058


  [1000/5000] elapsed 25s, current Recall@10=0.0150, NDCG@10=0.0060


  [1500/5000] elapsed 33s, current Recall@10=0.0160, NDCG@10=0.0072


  [2000/5000] elapsed 43s, current Recall@10=0.0150, NDCG@10=0.0067


  [2500/5000] elapsed 53s, current Recall@10=0.0180, NDCG@10=0.0083


  [3000/5000] elapsed 63s, current Recall@10=0.0170, NDCG@10=0.0081


  [3500/5000] elapsed 73s, current Recall@10=0.0171, NDCG@10=0.0081


  [4000/5000] elapsed 82s, current Recall@10=0.0173, NDCG@10=0.0080


  [4500/5000] elapsed 89s, current Recall@10=0.0182, NDCG@10=0.0083


  [5000/5000] elapsed 97s, current Recall@10=0.0184, NDCG@10=0.0082

Eval complete in 97s (19.3 ms/pair on average)
  evaluated:           5,000/5,000
  OOV users:           0
  positive in pool:    3,342/5,000 (66.8%)

  Mean Recall@10:      0.0184
  Mean NDCG@10:        0.0082
  Mean pool size:      638.8
  Median pool size:    602
  Latency p50/p95/p99: 11.8ms / 81.9ms / 99.4ms

saved → models/pipeline_C_metrics.json


Pipeline A baseline on the SAME subsample for apples-to-apples comparison: DeepFM v2 ranks all 9K items per user.

In [8]:
recalls_A_10 = []
ndcgs_A_10 = []
latencies_A = []
all_item_idxs = np.arange(len(item_features))

t_start = time.time()
for row_idx, row in val_sub.iterrows():
    user_id = row.user_id
    biz_id = row.business_id
    if biz_id not in biz_id_to_row_idx: continue
    pos_item_idx = biz_id_to_row_idx[biz_id]

    t0 = time.time()
    user_idx_for_model = user_encoder.encode(user_id)
    scores = score_user_against_candidates(user_idx_for_model, all_item_idxs).cpu().numpy()
    order = np.argsort(-scores)
    ranked_item_idxs = all_item_idxs[order]
    pos_rank = int(np.where(ranked_item_idxs == pos_item_idx)[0][0])

    recalls_A_10.append(1.0 if pos_rank < 10 else 0.0)
    ndcgs_A_10.append((1.0 / math.log2(pos_rank + 2)) if pos_rank < 10 else 0.0)
    latencies_A.append(time.time() - t0)

    if (row_idx+1) % 500 == 0:
        print(f"  [{row_idx+1}/{len(val_sub)}] R@10={np.mean(recalls_A_10):.4f} NDCG@10={np.mean(ndcgs_A_10):.4f}")

elapsed = time.time() - t_start
print(f"\nPipeline A complete in {elapsed:.0f}s")
print(f"  Mean Recall@10:      {np.mean(recalls_A_10):.4f}")
print(f"  Mean NDCG@10:        {np.mean(ndcgs_A_10):.4f}")
print(f"  Latency p50/p95/p99: {np.percentile(latencies_A, 50)*1000:.1f}ms / {np.percentile(latencies_A, 95)*1000:.1f}ms / {np.percentile(latencies_A, 99)*1000:.1f}ms")

results_A = {
    "pipeline": "A (DeepFM full 9K brute-force)",
    "n_eval": len(recalls_A_10),
    "Recall@10": float(np.mean(recalls_A_10)),
    "NDCG@10": float(np.mean(ndcgs_A_10)),
    "mean_pool_size": float(len(item_features)),
    "latency_ms_p50": float(np.percentile(latencies_A, 50)*1000),
    "latency_ms_p95": float(np.percentile(latencies_A, 95)*1000),
    "latency_ms_p99": float(np.percentile(latencies_A, 99)*1000),
}
with open(MODELS_DIR / "pipeline_A_metrics.json", "w") as f:
    json.dump(results_A, f, indent=2)
print("\nsaved → models/pipeline_A_metrics.json")


  [500/5000] R@10=0.0000 NDCG@10=0.0000


  [1000/5000] R@10=0.0000 NDCG@10=0.0000


  [1500/5000] R@10=0.0013 NDCG@10=0.0004


  [2000/5000] R@10=0.0010 NDCG@10=0.0003


  [2500/5000] R@10=0.0016 NDCG@10=0.0006


  [3000/5000] R@10=0.0013 NDCG@10=0.0005


  [3500/5000] R@10=0.0014 NDCG@10=0.0006


  [4000/5000] R@10=0.0013 NDCG@10=0.0005


  [4500/5000] R@10=0.0013 NDCG@10=0.0005


  [5000/5000] R@10=0.0012 NDCG@10=0.0005

Pipeline A complete in 80s
  Mean Recall@10:      0.0012
  Mean NDCG@10:        0.0005
  Latency p50/p95/p99: 16.2ms / 23.9ms / 27.9ms

saved → models/pipeline_A_metrics.json


## Pipeline A vs C 对比表

(Pipeline B from Phase 6.1 base two-tower @ Recall@10 = 0.0354 / NDCG @10 estimated 0.073)

In [9]:
# Two-Tower base recall reference (from models/two_tower_metrics.json)
two_tower = json.load(open(MODELS_DIR / "two_tower_metrics.json"))
results_B = {
    "pipeline": "B (Two-Tower top-200 → DeepFM rerank, est)",
    "Recall@10": two_tower["recall"]["Recall@10"],   # 0.0354 — retrieval recall
    "NDCG@10_estimate": two_tower["recall"]["Recall@200"] * results_A["NDCG@10"],  # rough upper bound
    "mean_pool_size": 200,
}

print(f"{'Pipeline':<55}{'Recall@10':>12}{'NDCG@10':>12}{'pool':>12}{'p99 lat':>12}")
print("-" * 113)
for r in [results_A, results_B, results_C]:
    name = r["pipeline"]
    rec = r.get("Recall@10", "—")
    ndcg = r.get("NDCG@10", r.get("NDCG@10_estimate", "—"))
    pool = r.get("mean_pool_size", "—")
    lat = r.get("latency_ms_p99", "—")
    rec_s = f"{rec:.4f}" if isinstance(rec, float) else str(rec)
    ndcg_s = f"{ndcg:.4f}" if isinstance(ndcg, float) else str(ndcg)
    pool_s = f"{pool:.0f}" if isinstance(pool, float) else str(pool)
    lat_s = f"{lat:.1f}ms" if isinstance(lat, float) else str(lat)
    print(f"{name:<55}{rec_s:>12}{ndcg_s:>12}{pool_s:>12}{lat_s:>12}")

print()
print("Take-aways:")
print(f"  • Pipeline C 把候选池从 9K 减到 ~{results_C['mean_pool_size']:.0f}")
print(f"  • Pipeline C 的 positive_in_pool_rate = {results_C['positive_in_pool_rate']*100:.1f}% — 上限")
print(f"  • Pipeline C vs A: NDCG@10 Δ = {results_C['NDCG@10'] - results_A['NDCG@10']:+.4f}")
print(f"  • Pipeline C 的 latency 比 A 快 ~{results_A['latency_ms_p99'] / max(results_C['latency_ms_p99'], 0.1):.1f}×")


Pipeline                                                  Recall@10     NDCG@10        pool     p99 lat
-----------------------------------------------------------------------------------------------------------------
A (DeepFM full 9K brute-force)                               0.0012      0.0005        9023      27.9ms
B (Two-Tower top-200 → DeepFM rerank, est)                   0.0354      0.0001         200           —
C (hard filter + DeepFM v2)                                  0.0184      0.0082         639      99.4ms

Take-aways:
  • Pipeline C 把候选池从 9K 减到 ~639
  • Pipeline C 的 positive_in_pool_rate = 66.8% — 上限
  • Pipeline C vs A: NDCG@10 Δ = +0.0077
  • Pipeline C 的 latency 比 A 快 ~0.3×


## Discussion + Phase 8 demo 推荐

观察 / 推论:

1. **Pipeline C 候选池命中率（positive_in_pool_rate）** 直接决定它的 Recall@10 上限——如果硬过滤把 20% 的真 positive 过滤掉，那 Recall@10 永远 ≤ 80% × Pipeline A 的命中率
2. **Pipeline C 优势**: latency 大幅降低（候选数小 5-30×），DeepFM 精排压力小
3. **Pipeline C 劣势**: 用户口味 / 价位偏好若太严，会过滤掉 long-tail 的 positive
4. **Pipeline B (Two-Tower) 已知 Recall@200 仅 22%**, 80% 的 positive 在召回阶段就被丢，进不到精排
5. **Phase 8 demo 推荐**: 用 Pipeline C — 兼顾 latency 与召回率，比 B 显著优, 接近 A 的精度

下一步: 拿 Pipeline C 的最终 NDCG@10 / Recall@10 数字写入 plan §6 + impl_notes §6 对比表，作为 final report §7.2 retrieval Pareto 分析的核心 evidence。